In [1]:
RUN_PROFILE = "kaggle_gpu_max_speed"   # "kaggle_gpu_max_speed", "balanced"

HF_DATASET_NAME = "common-pile/peS2o_filtered"
HF_SPLIT = "train"
SOURCE_SUBSTR = "s2orc"
MIN_WORDS = 500

PAIR_OUT_ROOT = "/kaggle/working/mmds_near_dup_optimized"
CACHE_DIR = f"{PAIR_OUT_ROOT}/cache"
BASE_JSONL_PATH = None

USE_FAST_HASH = True
USE_GPU_MINHASH = True
AUTO_INSTALL_CUPY = False   # bật True nếu bạn muốn thử GPU MinHash; install Cupy khá nặng
RUN_PAIR_DIAGNOSTIC = False # tắt mặc định để tối đa tốc độ; phần chính dùng stream eval
SAVE_SIGNATURE_CACHE = True
RANDOM_SEED = 42

if RUN_PROFILE == "balanced":
    TARGET_BASE_DOCS = 50_000
    TUNING_BASE_N = 4_000
    TEST_BASE_N = 16_000

    NUM_PARSER_VARIANTS = 4
    NUM_TRUNC_VARIANTS = 4

    PREVALENCE_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]
    TUNING_STREAM_DOCS = 6_000
    TEST_STREAM_DOCS = 20_000

    PAIR_TUNING_SOURCE_LIMIT = 800
    PAIR_TEST_SOURCE_LIMIT = 1_200

    THRESHOLD_GRID = [0.4, 0.6, 0.8]
    NUM_PERM_GRID = [64, 128]
    NGRAM_GRID = [3]
else:
    TARGET_BASE_DOCS = 30_000
    TUNING_BASE_N = 3_000
    TEST_BASE_N = 10_000

    NUM_PARSER_VARIANTS = 3
    NUM_TRUNC_VARIANTS = 3

    PREVALENCE_GRID = [0.1, 0.5, 0.9]
    TUNING_STREAM_DOCS = 4_000
    TEST_STREAM_DOCS = 12_000

    PAIR_TUNING_SOURCE_LIMIT = 500
    PAIR_TEST_SOURCE_LIMIT = 800

    THRESHOLD_GRID = [0.4, 0.6, 0.8]
    NUM_PERM_GRID = [64, 128]
    NGRAM_GRID = [3]


BASE_JSONL_PATH = f"{CACHE_DIR}/pes2o_base_{TARGET_BASE_DOCS}.jsonl"


In [2]:
!pip -q install datasets pyarrow fastparquet datasketch scikit-learn xxhash rank_bm25


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 8.0 MB/s eta 0:00:00


In [3]:
import importlib.util, subprocess, sys

if AUTO_INSTALL_CUPY and USE_GPU_MINHASH:
    if importlib.util.find_spec("cupy") is None:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
        except Exception as e:
            print("Cupy install failed, fallback CPU:", repr(e))


In [4]:
import gc
import io
import json
import os
import random
import re
import time
from bisect import bisect_left, bisect_right
from collections import defaultdict, deque
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from datasketch import MinHash, MinHashLSH
from rank_bm25 import BM25Okapi
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import xxhash

try:
    import cupy
    CUPY_AVAILABLE = True
except Exception:
    CUPY_AVAILABLE = False

Path(PAIR_OUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("RUN_PROFILE:", RUN_PROFILE)
print("CUPY_AVAILABLE:", CUPY_AVAILABLE)
print("USE_GPU_MINHASH:", USE_GPU_MINHASH)


RUN_PROFILE: kaggle_gpu_max_speed
CUPY_AVAILABLE: True
USE_GPU_MINHASH: True


In [5]:
_SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')
_TOKEN_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)

def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def good_doc(x):
    text = x.get("text", "")
    source = str(x.get("source", "")).lower()
    if not text or not isinstance(text, str):
        return False
    if SOURCE_SUBSTR not in source:
        return False
    if len(text.split()) < MIN_WORDS:
        return False
    return True

def split_sentences(text: str):
    sents = _SENT_SPLIT_RE.split(clean_text(text))
    return [s.strip() for s in sents if s.strip()]

def lexical_tokens(text: str):
    return _TOKEN_RE.findall(clean_text(text).lower())

def normalize_text_for_minhash(text: str) -> str:
    return clean_text(text).lower()

def tokenize_words(text: str):
    return _TOKEN_RE.findall(normalize_text_for_minhash(text))

def make_shingles_from_words(words, n=3):
    if not words:
        return [""]
    if n <= 1:
        # list unique để giảm update_batch đầu vào
        return list(dict.fromkeys(words))
    if len(words) < n:
        return [" ".join(words)]
    out = []
    seen = set()
    for i in range(len(words) - n + 1):
        sh = " ".join(words[i:i+n])
        if sh not in seen:
            out.append(sh)
            seen.add(sh)
    return out

def fast_hash32(b: bytes) -> int:
    return xxhash.xxh32_intdigest(b)

MINHASH_GPU_MODE = "detect" if USE_GPU_MINHASH else "disable"
MINHASH_HASHFUNC = fast_hash32 if USE_FAST_HASH else None
MINHASH_SEED = 1

def build_minhash_from_shingles(shingles, num_perm=64):
    kwargs = dict(num_perm=num_perm, seed=MINHASH_SEED, gpu_mode=MINHASH_GPU_MODE)
    if MINHASH_HASHFUNC is not None:
        kwargs["hashfunc"] = MINHASH_HASHFUNC
    mh = MinHash(**kwargs)
    mh.update_batch([s.encode("utf-8") for s in shingles])
    return mh

def compute_metrics(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


In [6]:
def load_or_collect_base_samples():
    if os.path.exists(BASE_JSONL_PATH):
        rows = []
        with open(BASE_JSONL_PATH, "r", encoding="utf-8") as f:
            for line in f:
                rows.append(json.loads(line))
        if len(rows) >= TARGET_BASE_DOCS:
            print("Loaded cached base samples:", len(rows))
            return rows[:TARGET_BASE_DOCS]
        print("Cached base samples insufficient, recollecting:", len(rows), "->", TARGET_BASE_DOCS)

    ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True)
    rows = []
    for item in ds:
        if good_doc(item):
            rows.append({
                "doc_id": str(item["id"]),
                "source": str(item.get("source", "")).lower(),
                "text": clean_text(item["text"]),
                "year": item.get("metadata", {}).get("year"),
            })
        if len(rows) >= TARGET_BASE_DOCS:
            break

    with open(BASE_JSONL_PATH, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Collected and cached base samples:", len(rows))
    return rows

samples = load_or_collect_base_samples()
print("Collected:", len(samples))


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/91 [00:00<?, ?it/s]

Collected and cached base samples: 30000
Collected: 30000


In [7]:
random.seed(RANDOM_SEED)
random.shuffle(samples)

tuning_base = samples[:TUNING_BASE_N]
test_base = samples[TUNING_BASE_N:TUNING_BASE_N + TEST_BASE_N]

print("tuning_base:", len(tuning_base))
print("test_base  :", len(test_base))
print("first doc_id:", tuning_base[0]["doc_id"])
print("first preview:", tuning_base[0]["text"][:250])


tuning_base: 3000
test_base  : 10000
first doc_id: 55903258
first preview: Scattering from ramified polymeric systems Here, of great interest to us is a quantitative study of the scattering properties from ramified polymeric systems of arbitrary topology. We consider three types of systems, namely ramified polymers in solut


In [8]:
def random_char_noise(token: str, rng: random.Random):
    if len(token) < 4:
        return token
    swaps = {"O": "0", "0": "O", "l": "1", "1": "l", "m": "rn", "rn": "m"}
    if rng.random() < 0.35:
        for a, b in swaps.items():
            if a in token:
                return token.replace(a, b, 1)
    return token

def make_parser_variant(text: str, rng: random.Random) -> str:
    text = clean_text(text)
    if not text:
        return text

    m = re.search(
        r'\b(references|bibliography|appendix|acknowledg(?:e)?ments?)\b',
        text,
        flags=re.IGNORECASE
    )
    if m and m.start() > len(text) * 0.45 and rng.random() < 0.55:
        text = text[:m.start()].strip()

    sents = split_sentences(text)
    if len(sents) < 8:
        return text

    kept = []
    for s in sents:
        n_w = len(s.split())
        if 4 <= n_w <= 15 and rng.random() < 0.12:
            continue
        kept.append(s)
    sents = kept if kept else sents

    rebuilt = []
    for s in sents:
        r = rng.random()
        if r < 0.10:
            rebuilt.append(s + "\n")
        elif r < 0.20:
            rebuilt.append(s + "\n\n")
        else:
            rebuilt.append(s + " ")
    text = "".join(rebuilt).strip()

    words = text.split()
    long_idxs = [i for i, w in enumerate(words) if len(w) >= 8]
    if long_idxs and rng.random() < 0.65:
        n_changes = max(1, min(len(long_idxs), len(words) // 180))
        for idx in rng.sample(long_idxs, k=n_changes):
            w = words[idx]
            cut = max(3, len(w) // 2)
            if rng.random() < 0.5:
                words[idx] = w[:cut] + "-\n" + w[cut:]
            else:
                words[idx] = w.replace("-", "")
    text = " ".join(words)

    words = text.split()
    candidate_idxs = [i for i, w in enumerate(words) if any(ch in w for ch in "O0l1m")]
    if candidate_idxs and rng.random() < 0.55:
        n_changes = max(1, min(len(candidate_idxs), len(words) // 220))
        for idx in rng.sample(candidate_idxs, k=n_changes):
            words[idx] = random_char_noise(words[idx], rng)
    text = " ".join(words)

    text = re.sub(r'\[(\d+(?:,\s*\d+)*)\]', lambda m: "(" + m.group(1) + ")", text)
    text = re.sub(r'\((\d+(?:,\s*\d+)*)\)', lambda m: "[" + m.group(1) + "]" if rng.random() < 0.35 else m.group(0), text)

    text = re.sub(r'([,;:])\s+', r'\1  ', text)
    text = re.sub(r'\n +', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    return text

def make_truncate_variant(text: str, rng: random.Random):
    text = clean_text(text)
    sents = split_sentences(text)
    n = len(sents)

    if n < 20:
        return text, "truncate_none"

    keep_ratio = rng.uniform(0.65, 0.85)
    remove_n = max(1, int(round(n * (1 - keep_ratio))))
    mode = rng.choice(["head", "tail", "middle", "two_gaps", "three_gaps"])

    if mode == "head":
        new_sents = sents[remove_n:]
        variant_type = "truncate_head"
    elif mode == "tail":
        new_sents = sents[:-remove_n]
        variant_type = "truncate_tail"
    elif mode == "middle":
        start_lo = max(1, n // 7)
        start_hi = max(start_lo, n - remove_n - 1)
        start = rng.randint(start_lo, start_hi)
        end = min(n, start + remove_n)
        new_sents = sents[:start] + sents[end:]
        variant_type = "truncate_middle"
    elif mode == "two_gaps":
        gap1 = max(1, remove_n // 2)
        gap2 = max(1, remove_n - gap1)

        start1 = rng.randint(max(1, n // 10), max(1, n // 3 - gap1))
        tmp = sents[:start1] + sents[start1 + gap1:]

        n2 = len(tmp)
        if n2 <= gap2 + 2:
            new_sents = tmp
        else:
            start2 = rng.randint(max(1, n2 // 2), max(1, n2 - gap2 - 1))
            new_sents = tmp[:start2] + tmp[start2 + gap2:]
        variant_type = "truncate_two_gaps"
    else:
        remaining = remove_n
        tmp = list(sents)
        for frac in [0.15, 0.45, 0.70]:
            if remaining <= 0 or len(tmp) < 6:
                break
            gap = max(1, remaining // 3)
            anchor = int(len(tmp) * frac)
            start = min(max(1, anchor), max(1, len(tmp) - gap - 1))
            tmp = tmp[:start] + tmp[start + gap:]
            remaining -= gap
        new_sents = tmp
        variant_type = "truncate_three_gaps"

    new_text = " ".join(new_sents).strip()
    return new_text if new_text else text, variant_type

def build_doc_benchmark(
    base_docs,
    split_name,
    seed=42,
    num_parser_variants=NUM_PARSER_VARIANTS,
    num_trunc_variants=NUM_TRUNC_VARIANTS,
):
    rng = random.Random(seed)
    rows = []

    for row in base_docs:
        source_id = str(row["doc_id"])
        text = clean_text(row["text"])

        rows.append({
            "doc_id": source_id,
            "source_doc_id": source_id,
            "split": split_name,
            "variant_family": "original",
            "variant_type": "original",
            "text": text,
            "n_words": len(text.split())
        })

        for i in range(num_parser_variants):
            parser_text = make_parser_variant(text, rng)
            rows.append({
                "doc_id": f"{source_id}_p{i}",
                "source_doc_id": source_id,
                "split": split_name,
                "variant_family": "parser",
                "variant_type": f"parser_noise_v{i}",
                "text": parser_text,
                "n_words": len(parser_text.split())
            })

        for i in range(num_trunc_variants):
            trunc_text, trunc_type = make_truncate_variant(text, rng)
            rows.append({
                "doc_id": f"{source_id}_t{i}",
                "source_doc_id": source_id,
                "split": split_name,
                "variant_family": "truncation",
                "variant_type": f"{trunc_type}_v{i}",
                "text": trunc_text,
                "n_words": len(trunc_text.split())
            })

    return pd.DataFrame(rows)

tuning_docs = build_doc_benchmark(tuning_base, "tuning", seed=42)
test_docs = build_doc_benchmark(test_base, "test", seed=43)

print("tuning_docs:", tuning_docs.shape)
print("test_docs  :", test_docs.shape)


tuning_docs: (21000, 7)
test_docs  : (70000, 7)


In [9]:
def sample_source_subset(doc_df, source_limit, seed=42):
    rng = random.Random(seed)
    source_ids = sorted(doc_df["source_doc_id"].astype(str).unique().tolist())
    rng.shuffle(source_ids)
    keep = set(source_ids[:min(source_limit, len(source_ids))])
    return doc_df[doc_df["source_doc_id"].astype(str).isin(keep)].copy()

def build_positive_pairs(doc_df):
    rows = []
    for source_doc_id, group in doc_df.groupby("source_doc_id"):
        recs = group[["doc_id", "variant_family"]].to_dict("records")
        for a, b in combinations(recs, 2):
            fams = sorted([a["variant_family"], b["variant_family"]])
            rows.append({
                "doc_id_1": a["doc_id"],
                "doc_id_2": b["doc_id"],
                "label": 1,
                "pair_family": "positive",
                "pair_type": f"{fams[0]}__{fams[1]}"
            })
    return pd.DataFrame(rows)

def jaccard_unigram_from_sets(sa, sb):
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def build_negative_pairs_bm25(
    doc_df,
    n_neg_per_pos=1,
    seed=42,
    bm25_top_k=20,
    hard_jaccard_min=0.20,
    hard_jaccard_max=0.48,
):
    rng = random.Random(seed)

    all_docs = doc_df[["doc_id", "source_doc_id", "variant_family", "n_words", "text"]].copy()
    originals = all_docs[all_docs["variant_family"] == "original"].copy().reset_index(drop=True)

    all_records = all_docs.to_dict("records")
    orig_records = originals.to_dict("records")

    source_to_variants = defaultdict(list)
    for r in all_records:
        source_to_variants[str(r["source_doc_id"])].append(r)
    source_ids = list(source_to_variants.keys())

    length_pairs = sorted(
        [(int(r["n_words"]), str(r["source_doc_id"])) for r in orig_records],
        key=lambda x: x[0]
    )
    sorted_lengths = [x[0] for x in length_pairs]
    sorted_source_ids = [x[1] for x in length_pairs]

    medium_source_cache = {}
    for r in orig_records:
        sid = str(r["source_doc_id"])
        n_words = int(r["n_words"])
        lo = bisect_left(sorted_lengths, int(0.7 * n_words))
        hi = bisect_right(sorted_lengths, int(1.3 * n_words))
        cands = [x for x in sorted_source_ids[lo:hi] if x != sid]
        medium_source_cache[sid] = cands

    bm25_corpus = [lexical_tokens(r["text"]) for r in orig_records]
    bm25 = BM25Okapi(bm25_corpus)

    orig_source_ids = [str(r["source_doc_id"]) for r in orig_records]
    orig_lengths = [int(r["n_words"]) for r in orig_records]
    orig_token_sets = [set(toks) for toks in bm25_corpus]

    hard_source_cache = {}
    for i, r in enumerate(orig_records):
        anchor_sid = str(r["source_doc_id"])
        anchor_len = int(r["n_words"])
        query_tokens = bm25_corpus[i]

        scores = bm25.get_scores(query_tokens)
        ranked_idx = np.argsort(scores)[::-1]

        hard_sources = []
        seen_sid = set()

        for j in ranked_idx:
            if j == i:
                continue
            other_sid = orig_source_ids[j]
            if other_sid == anchor_sid or other_sid in seen_sid:
                continue

            other_len = orig_lengths[j]
            if not (0.8 * anchor_len <= other_len <= 1.25 * anchor_len):
                continue

            jac = jaccard_unigram_from_sets(orig_token_sets[i], orig_token_sets[j])
            if not (hard_jaccard_min <= jac <= hard_jaccard_max):
                continue

            hard_sources.append(other_sid)
            seen_sid.add(other_sid)
            if len(hard_sources) >= bm25_top_k:
                break

        hard_source_cache[anchor_sid] = hard_sources

    cluster_sizes = doc_df.groupby("source_doc_id").size()
    n_pos = int((cluster_sizes * (cluster_sizes - 1) // 2).sum())
    total_neg = n_pos * n_neg_per_pos

    rows = []
    seen_pairs = set()
    difficulty_cycle = ["hard", "medium", "easy"]
    attempts = 0
    max_attempts = max(2000, total_neg * 40)

    while len(rows) < total_neg and attempts < max_attempts:
        attempts += 1
        difficulty = difficulty_cycle[len(rows) % len(difficulty_cycle)]

        anchor = rng.choice(all_records)
        anchor_doc_id = str(anchor["doc_id"])
        anchor_sid = str(anchor["source_doc_id"])
        anchor_len = int(anchor["n_words"])

        neg = None
        pair_family = None

        if difficulty == "hard":
            hard_sources = hard_source_cache.get(anchor_sid, [])
            if hard_sources:
                top_sources = hard_sources[:8].copy()
                rng.shuffle(top_sources)
                for neg_sid in top_sources:
                    pool = source_to_variants[neg_sid]
                    length_ok = [
                        x for x in pool
                        if 0.75 * anchor_len <= int(x["n_words"]) <= 1.35 * anchor_len
                    ]
                    neg = rng.choice(length_ok if length_ok else pool)
                    pair_family = "negative_hard_bm25_jaccard"
                    break

        if neg is None and difficulty in {"hard", "medium"}:
            medium_sources = medium_source_cache.get(anchor_sid, [])
            if medium_sources:
                neg_sid = rng.choice(medium_sources)
                pool = source_to_variants[neg_sid]
                length_ok = [
                    x for x in pool
                    if 0.75 * anchor_len <= int(x["n_words"]) <= 1.35 * anchor_len
                ]
                neg = rng.choice(length_ok if length_ok else pool)
                pair_family = "negative_medium_length"

        if neg is None:
            if len(source_ids) <= 1:
                continue
            neg_sid = rng.choice(source_ids)
            while neg_sid == anchor_sid:
                neg_sid = rng.choice(source_ids)
            neg = rng.choice(source_to_variants[neg_sid])
            pair_family = "negative_easy_random"

        neg_doc_id = str(neg["doc_id"])
        if neg_doc_id == anchor_doc_id:
            continue

        pair_key = tuple(sorted([anchor_doc_id, neg_doc_id]))
        if pair_key in seen_pairs:
            continue

        seen_pairs.add(pair_key)
        rows.append({
            "doc_id_1": anchor_doc_id,
            "doc_id_2": neg_doc_id,
            "label": 0,
            "pair_family": pair_family,
            "pair_type": pair_family
        })

    if len(rows) < total_neg:
        print(f"Warning: requested {total_neg} negatives but built {len(rows)} unique negatives.")

    return pd.DataFrame(rows)


In [10]:
def build_stream_eval_dataset(doc_df, prevalence, total_docs, seed=42):
    rng = random.Random(seed)

    source_to_group = {}
    for sid, group in doc_df.groupby("source_doc_id"):
        g = group.copy()
        originals = g[g["variant_family"] == "original"]
        dups = g[g["variant_family"] != "original"]

        if len(originals) == 0 or len(dups) == 0:
            continue

        source_to_group[str(sid)] = {
            "original": originals.iloc[0].to_dict(),
            "duplicates": dups.to_dict("records")
        }

    available_sources = list(source_to_group.keys())
    rng.shuffle(available_sources)

    n_unique = max(1, int(round(total_docs * (1 - prevalence))))
    n_unique = min(n_unique, len(available_sources))
    n_dups_target = total_docs - n_unique
    selected_sources = available_sources[:n_unique]

    originals = []
    dup_pool = {}
    for sid in selected_sources:
        originals.append(source_to_group[sid]["original"])
        local_dups = source_to_group[sid]["duplicates"].copy()
        rng.shuffle(local_dups)
        dup_pool[sid] = deque(local_dups)

    max_possible_dups = sum(len(v) for v in dup_pool.values())
    if n_dups_target > max_possible_dups:
        n_dups_target = max_possible_dups
        total_docs = n_unique + n_dups_target

    selected_dups = []
    active_sources = selected_sources.copy()
    rng.shuffle(active_sources)

    while len(selected_dups) < n_dups_target and active_sources:
        next_active = []
        for sid in active_sources:
            if len(selected_dups) >= n_dups_target:
                break
            if dup_pool[sid]:
                selected_dups.append(dup_pool[sid].popleft())
            if dup_pool[sid]:
                next_active.append(sid)
        rng.shuffle(next_active)
        active_sources = next_active

    rng.shuffle(originals)
    pending = defaultdict(list)
    for row in selected_dups:
        pending[str(row["source_doc_id"])].append(row)

    original_queue = deque(originals)
    seen_sources = []
    stream_rows = []

    while original_queue:
        orig = original_queue.popleft()
        sid = str(orig["source_doc_id"])
        stream_rows.append(orig)
        seen_sources.append(sid)

        if seen_sources and rng.random() < 0.75:
            k_release = rng.choice([0, 1, 1, 2])
            release_candidates = seen_sources.copy()
            rng.shuffle(release_candidates)

            released = 0
            for rsid in release_candidates:
                if released >= k_release:
                    break
                if pending[rsid]:
                    stream_rows.append(pending[rsid].pop())
                    released += 1

    leftovers = []
    for sid in list(pending.keys()):
        leftovers.extend(pending[sid])
    rng.shuffle(leftovers)
    stream_rows.extend(leftovers)

    stream_df = pd.DataFrame(stream_rows).reset_index(drop=True)
    stream_df["duplicate_prevalence_target"] = prevalence
    stream_df["stream_pos"] = np.arange(len(stream_df))
    return stream_df

def build_streams_for_grid(doc_df, prevalence_grid, total_docs, seed_base):
    streams = []
    union_doc_ids = set()
    for i, prevalence in enumerate(prevalence_grid):
        sdf = build_stream_eval_dataset(
            doc_df=doc_df,
            prevalence=prevalence,
            total_docs=total_docs,
            seed=seed_base + i,
        )
        streams.append((prevalence, sdf))
        union_doc_ids.update(sdf["doc_id"].astype(str).tolist())
    return streams, union_doc_ids


In [11]:
class SignatureCache:
    def __init__(self, doc_df, doc_ids, ngram_n, num_perm, cache_name):
        self.doc_df = doc_df[doc_df["doc_id"].astype(str).isin(set(doc_ids))][["doc_id", "text"]].copy()
        self.doc_df["doc_id"] = self.doc_df["doc_id"].astype(str)
        self.doc_df = self.doc_df.sort_values("doc_id").reset_index(drop=True)

        self.doc_ids = self.doc_df["doc_id"].tolist()
        self.row_map = {doc_id: i for i, doc_id in enumerate(self.doc_ids)}
        self.ngram_n = ngram_n
        self.num_perm = num_perm
        self.cache_name = cache_name

        base_kwargs = dict(num_perm=num_perm, seed=MINHASH_SEED, gpu_mode=MINHASH_GPU_MODE)
        if MINHASH_HASHFUNC is not None:
            base_kwargs["hashfunc"] = MINHASH_HASHFUNC
        base_mh = MinHash(**base_kwargs)
        self.permutations = base_mh.permutations

        self.sig_matrix = None
        self.obj_cache = {}

    @property
    def npz_path(self):
        return Path(CACHE_DIR) / f"{self.cache_name}_ng{self.ngram_n}_perm{self.num_perm}.npz"

    def load_or_build(self):
        if SAVE_SIGNATURE_CACHE and self.npz_path.exists():
            data = np.load(self.npz_path, allow_pickle=True)
            cached_doc_ids = data["doc_ids"].tolist()
            if cached_doc_ids == self.doc_ids:
                self.sig_matrix = data["sig_matrix"]
                print("Loaded signature cache:", self.npz_path.name, self.sig_matrix.shape)
                return

        t0 = time.perf_counter()
        sig_matrix = np.zeros((len(self.doc_ids), self.num_perm), dtype=np.uint64)

        for i, row in enumerate(self.doc_df.itertuples(index=False)):
            words = tokenize_words(row.text)
            shingles = make_shingles_from_words(words, n=self.ngram_n)
            mh = build_minhash_from_shingles(shingles, num_perm=self.num_perm)
            sig_matrix[i] = mh.hashvalues

            if (i + 1) % 1000 == 0 or (i + 1) == len(self.doc_ids):
                print(f"[sig-cache] {self.cache_name} ng={self.ngram_n} perm={self.num_perm}: {i+1}/{len(self.doc_ids)}")

        self.sig_matrix = sig_matrix

        if SAVE_SIGNATURE_CACHE:
            np.savez_compressed(self.npz_path, doc_ids=np.asarray(self.doc_ids, dtype=object), sig_matrix=self.sig_matrix)

        print("Built signature cache in", round(time.perf_counter() - t0, 2), "sec")

    def get_mh(self, doc_id):
        doc_id = str(doc_id)
        mh = self.obj_cache.get(doc_id)
        if mh is not None:
            return mh

        idx = self.row_map[doc_id]
        kwargs = dict(
            num_perm=self.num_perm,
            seed=MINHASH_SEED,
            hashvalues=self.sig_matrix[idx],
            permutations=self.permutations,
        )
        if MINHASH_HASHFUNC is not None:
            kwargs["hashfunc"] = MINHASH_HASHFUNC

        mh = MinHash(**kwargs)
        self.obj_cache[doc_id] = mh
        return mh


In [12]:
def run_minhash_lsh_stream_eval_precomputed(stream_df, sig_cache, threshold=0.5):
    lsh = MinHashLSH(threshold=threshold, num_perm=sig_cache.num_perm)

    seen_source_ids = set()
    y_true = []
    y_pred = []

    query_sec = 0.0
    insert_sec = 0.0
    total_hits = 0
    docs_with_hits = 0

    for row in stream_df[["doc_id", "source_doc_id"]].itertuples(index=False):
        doc_id = str(row.doc_id)
        sid = str(row.source_doc_id)

        y_true.append(1 if sid in seen_source_ids else 0)
        mh = sig_cache.get_mh(doc_id)

        t0 = time.perf_counter()
        hits = lsh.query(mh)
        query_sec += time.perf_counter() - t0

        pred = 1 if len(hits) > 0 else 0
        y_pred.append(pred)

        if hits:
            docs_with_hits += 1
            total_hits += len(hits)

        t0 = time.perf_counter()
        lsh.insert(doc_id, mh)
        insert_sec += time.perf_counter() - t0

        seen_source_ids.add(sid)

    y_true = np.asarray(y_true, dtype=np.int32)
    y_pred = np.asarray(y_pred, dtype=np.int32)

    metrics = compute_metrics(y_true, y_pred)
    metrics.update({
        "n_docs": int(len(stream_df)),
        "n_duplicates_true": int(y_true.sum()),
        "n_predicted_duplicates": int(y_pred.sum()),
        "docs_with_hits": int(docs_with_hits),
        "total_hit_count": int(total_hits),
        "query_sec": float(query_sec),
        "insert_sec": float(insert_sec),
    })
    return metrics, y_true, y_pred

def evaluate_prevalence_grid_from_streams(streams, sig_cache, threshold):
    rows = []
    for prevalence, stream_df in streams:
        metrics, y_true, y_pred = run_minhash_lsh_stream_eval_precomputed(
            stream_df=stream_df,
            sig_cache=sig_cache,
            threshold=threshold,
        )

        rows.append({
            "prevalence": prevalence,
            "ngram_n": sig_cache.ngram_n,
            "num_perm": sig_cache.num_perm,
            "threshold": threshold,
            "build_signature_sec": np.nan,
            **metrics
        })

        print(
            f"[STREAM] prev={prevalence:.1f} | ng={sig_cache.ngram_n} | perm={sig_cache.num_perm} | thr={threshold:.2f} "
            f"-> P={metrics['precision']:.4f} R={metrics['recall']:.4f} F1={metrics['f1']:.4f}"
        )
    return pd.DataFrame(rows)


In [13]:
tuning_streams, tuning_stream_doc_ids = build_streams_for_grid(
    doc_df=tuning_docs,
    prevalence_grid=PREVALENCE_GRID,
    total_docs=TUNING_STREAM_DOCS,
    seed_base=5000,
)

test_streams, test_stream_doc_ids = build_streams_for_grid(
    doc_df=test_docs,
    prevalence_grid=PREVALENCE_GRID,
    total_docs=TEST_STREAM_DOCS,
    seed_base=9000,
)

print("tuning stream count:", len(tuning_streams), "| unique docs used:", len(tuning_stream_doc_ids))
print("test stream count  :", len(test_streams), "| unique docs used:", len(test_stream_doc_ids))


tuning stream count: 3 | unique docs used: 7918
test stream count  : 3 | unique docs used: 24090


In [14]:
stream_tuning_rows = []
tuning_sig_caches = {}

for ngram_n in NGRAM_GRID:
    for num_perm in NUM_PERM_GRID:
        cache_key = ("tuning", ngram_n, num_perm)
        sig_cache = SignatureCache(
            doc_df=tuning_docs,
            doc_ids=tuning_stream_doc_ids,
            ngram_n=ngram_n,
            num_perm=num_perm,
            cache_name="tuning_stream",
        )
        sig_cache.load_or_build()
        tuning_sig_caches[cache_key] = sig_cache

        for threshold in THRESHOLD_GRID:
            prev_df = evaluate_prevalence_grid_from_streams(
                streams=tuning_streams,
                sig_cache=sig_cache,
                threshold=threshold,
            )

            summary = {
                "ngram_n": ngram_n,
                "num_perm": num_perm,
                "threshold": threshold,
                "mean_precision": float(prev_df["precision"].mean()),
                "mean_recall": float(prev_df["recall"].mean()),
                "mean_f1": float(prev_df["f1"].mean()),
                "min_f1": float(prev_df["f1"].min()),
                "max_f1": float(prev_df["f1"].max()),
                "mean_query_sec": float(prev_df["query_sec"].mean()),
                "mean_insert_sec": float(prev_df["insert_sec"].mean()),
            }
            stream_tuning_rows.append(summary)

stream_tuning_df = pd.DataFrame(stream_tuning_rows).sort_values(
    ["mean_f1", "mean_precision", "mean_recall"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("\n=== Stream tuning top 10 ===")
display(stream_tuning_df.head(10))

best_stream_cfg = stream_tuning_df.iloc[0].to_dict()
print("\n=== Best stream config ===")
print(best_stream_cfg)


[sig-cache] tuning_stream ng=3 perm=64: 1000/7918
[sig-cache] tuning_stream ng=3 perm=64: 2000/7918
[sig-cache] tuning_stream ng=3 perm=64: 3000/7918
[sig-cache] tuning_stream ng=3 perm=64: 4000/7918
[sig-cache] tuning_stream ng=3 perm=64: 5000/7918
[sig-cache] tuning_stream ng=3 perm=64: 6000/7918
[sig-cache] tuning_stream ng=3 perm=64: 7000/7918
[sig-cache] tuning_stream ng=3 perm=64: 7918/7918
Built signature cache in 37.69 sec
[STREAM] prev=0.1 | ng=3 | perm=64 | thr=0.40 -> P=1.0000 R=0.9930 F1=0.9965
[STREAM] prev=0.5 | ng=3 | perm=64 | thr=0.40 -> P=1.0000 R=0.9935 F1=0.9967
[STREAM] prev=0.9 | ng=3 | perm=64 | thr=0.40 -> P=1.0000 R=0.9958 F1=0.9979
[STREAM] prev=0.1 | ng=3 | perm=64 | thr=0.60 -> P=1.0000 R=0.9180 F1=0.9572
[STREAM] prev=0.5 | ng=3 | perm=64 | thr=0.60 -> P=1.0000 R=0.9250 F1=0.9610
[STREAM] prev=0.9 | ng=3 | perm=64 | thr=0.60 -> P=1.0000 R=0.9513 F1=0.9750
[STREAM] prev=0.1 | ng=3 | perm=64 | thr=0.80 -> P=1.0000 R=0.6520 F1=0.7893
[STREAM] prev=0.5 | ng=3 |

,ngram_n,num_perm,threshold,mean_precision,mean_recall,mean_f1,min_f1,max_f1,mean_query_sec,mean_insert_sec
0,3,128,0.4,1.0,0.999333,0.999666,0.999500,1.000000,0.101638,0.241397
1,3,64,0.4,1.0,0.994111,0.997046,0.996488,0.997912,0.054737,0.113165
2,3,128,0.6,1.0,0.953111,0.975969,0.970664,0.982620,0.063659,0.226735
3,3,64,0.6,1.0,0.931417,0.964434,0.957247,0.975016,0.095945,0.044204
4,3,128,0.8,1.0,0.698556,0.822235,0.808105,0.848369,0.035963,0.041150
5,3,64,0.8,1.0,0.677917,0.807604,0.789346,0.839739,0.022202,0.025173



=== Best stream config ===
{'ngram_n': 3.0, 'num_perm': 128.0, 'threshold': 0.4, 'mean_precision': 1.0, 'mean_recall': 0.9993333333333334, 'mean_f1': 0.9996664999166249, 'min_f1': 0.9994997498749375, 'max_f1': 1.0, 'mean_query_sec': 0.10163800333013266, 'mean_insert_sec': 0.241396717334131}


In [15]:
best_stream_ngram_n = int(best_stream_cfg["ngram_n"])
best_stream_num_perm = int(best_stream_cfg["num_perm"])
best_stream_threshold = float(best_stream_cfg["threshold"])

test_sig_cache = SignatureCache(
    doc_df=test_docs,
    doc_ids=test_stream_doc_ids,
    ngram_n=best_stream_ngram_n,
    num_perm=best_stream_num_perm,
    cache_name="test_stream",
)
test_sig_cache.load_or_build()

stream_test_df = evaluate_prevalence_grid_from_streams(
    streams=test_streams,
    sig_cache=test_sig_cache,
    threshold=best_stream_threshold,
)

print("\n=== Stream test results ===")
display(stream_test_df)

print("\n=== Stream test summary ===")
print(stream_test_df[["precision", "recall", "f1"]].mean())


[sig-cache] test_stream ng=3 perm=128: 1000/24090
[sig-cache] test_stream ng=3 perm=128: 2000/24090
[sig-cache] test_stream ng=3 perm=128: 3000/24090
[sig-cache] test_stream ng=3 perm=128: 4000/24090
[sig-cache] test_stream ng=3 perm=128: 5000/24090
[sig-cache] test_stream ng=3 perm=128: 6000/24090
[sig-cache] test_stream ng=3 perm=128: 7000/24090
[sig-cache] test_stream ng=3 perm=128: 8000/24090
[sig-cache] test_stream ng=3 perm=128: 9000/24090
[sig-cache] test_stream ng=3 perm=128: 10000/24090
[sig-cache] test_stream ng=3 perm=128: 11000/24090
[sig-cache] test_stream ng=3 perm=128: 12000/24090
[sig-cache] test_stream ng=3 perm=128: 13000/24090
[sig-cache] test_stream ng=3 perm=128: 14000/24090
[sig-cache] test_stream ng=3 perm=128: 15000/24090
[sig-cache] test_stream ng=3 perm=128: 16000/24090
[sig-cache] test_stream ng=3 perm=128: 17000/24090
[sig-cache] test_stream ng=3 perm=128: 18000/24090
[sig-cache] test_stream ng=3 perm=128: 19000/24090
[sig-cache] test_stream ng=3 perm=128: 2

,prevalence,ngram_n,num_perm,threshold,build_signature_sec,precision,recall,f1,n_docs,n_duplicates_true,n_predicted_duplicates,docs_with_hits,total_hit_count,query_sec,insert_sec
0,0.1,3,128,0.4,NaN,1.0,0.999000,0.999500,12000,2000,1998,1998,1998,0.572727,0.840704
1,0.5,3,128,0.4,NaN,1.0,0.998333,0.999166,12000,6000,5990,5990,5990,0.360560,0.975211
2,0.9,3,128,0.4,NaN,1.0,0.999444,0.999722,8400,7200,7196,7196,24964,0.267940,0.597020



=== Stream test summary ===
precision    1.000000
recall       0.998926
f1           0.999463
dtype: float64


In [16]:
pair_tuning_df = None
pair_test_result = None
pair_family_report = None
positive_pair_type_report = None

def build_lsh_and_candidate_pairs_from_sigcache(sig_cache, doc_ids, threshold=0.5):
    lsh = MinHashLSH(threshold=threshold, num_perm=sig_cache.num_perm)

    t0 = time.perf_counter()
    for doc_id in doc_ids:
        lsh.insert(str(doc_id), sig_cache.get_mh(str(doc_id)))
    insert_sec = time.perf_counter() - t0

    t1 = time.perf_counter()
    candidate_pairs = set()
    for doc_id in doc_ids:
        mh = sig_cache.get_mh(str(doc_id))
        hits = lsh.query(mh)
        for other_id in hits:
            if other_id == doc_id:
                continue
            candidate_pairs.add(tuple(sorted((str(doc_id), str(other_id)))))
    query_sec = time.perf_counter() - t1
    return candidate_pairs, insert_sec, query_sec

def predict_pairs_from_candidates(pair_df, candidate_pairs):
    y_true = pair_df["label"].astype(int).to_numpy()
    keys = [
        tuple(sorted((str(a), str(b))))
        for a, b in zip(pair_df["doc_id_1"].values, pair_df["doc_id_2"].values)
    ]
    y_pred = np.array([1 if k in candidate_pairs else 0 for k in keys], dtype=np.int32)
    return y_true, y_pred

def analyze_pair_family_better(pair_df, y_pred):
    tmp = pair_df.copy()
    tmp["predicted"] = y_pred
    rows = []

    for family, sub in tmp.groupby("pair_family"):
        y_true_f = sub["label"].astype(int).to_numpy()
        y_pred_f = sub["predicted"].astype(int).to_numpy()
        row = {
            "pair_family": family,
            "n_pairs": len(sub),
            "accuracy": float((y_true_f == y_pred_f).mean()),
            "pred_positive_rate": float(y_pred_f.mean()),
        }
        if y_true_f.sum() > 0:
            row["precision"] = precision_score(y_true_f, y_pred_f, zero_division=0)
            row["recall"] = recall_score(y_true_f, y_pred_f, zero_division=0)
            row["f1"] = f1_score(y_true_f, y_pred_f, zero_division=0)
        else:
            row["false_positive_rate"] = float(y_pred_f.mean())
        rows.append(row)

    return pd.DataFrame(rows).sort_values("pair_family").reset_index(drop=True)

def analyze_positive_pair_types(pair_df, y_pred):
    tmp = pair_df.copy()
    tmp["predicted"] = y_pred
    pos_only = tmp[tmp["label"] == 1].copy()
    rows = []
    for ptype, sub in pos_only.groupby("pair_type"):
        y_true = sub["label"].astype(int).to_numpy()
        y_pred_sub = sub["predicted"].astype(int).to_numpy()
        rows.append({
            "pair_type": ptype,
            "n_pairs": len(sub),
            "recall": recall_score(y_true, y_pred_sub, zero_division=0),
            "pred_pos_rate": float(y_pred_sub.mean()),
        })
    return pd.DataFrame(rows).sort_values("pair_type").reset_index(drop=True)

if RUN_PAIR_DIAGNOSTIC:
    tuning_pair_docs = sample_source_subset(tuning_docs, PAIR_TUNING_SOURCE_LIMIT, seed=42)
    test_pair_docs = sample_source_subset(test_docs, PAIR_TEST_SOURCE_LIMIT, seed=43)

    tuning_pos = build_positive_pairs(tuning_pair_docs)
    tuning_neg = build_negative_pairs_bm25(tuning_pair_docs, n_neg_per_pos=1, seed=42)
    test_pos = build_positive_pairs(test_pair_docs)
    test_neg = build_negative_pairs_bm25(test_pair_docs, n_neg_per_pos=1, seed=43)

    tuning_pairs = pd.concat([tuning_pos, tuning_neg], ignore_index=True)
    test_pairs = pd.concat([test_pos, test_neg], ignore_index=True)

    pair_tuning_rows = []
    pair_sig_caches = {}

    for ngram_n in NGRAM_GRID:
        for num_perm in NUM_PERM_GRID:
            pair_doc_ids = tuning_pair_docs["doc_id"].astype(str).tolist()
            sig_cache = SignatureCache(
                doc_df=tuning_pair_docs,
                doc_ids=pair_doc_ids,
                ngram_n=ngram_n,
                num_perm=num_perm,
                cache_name="tuning_pair",
            )
            sig_cache.load_or_build()
            pair_sig_caches[(ngram_n, num_perm)] = sig_cache

            for threshold in THRESHOLD_GRID:
                candidate_pairs, insert_sec, query_sec = build_lsh_and_candidate_pairs_from_sigcache(
                    sig_cache=sig_cache,
                    doc_ids=pair_doc_ids,
                    threshold=threshold,
                )
                y_true, y_pred = predict_pairs_from_candidates(tuning_pairs, candidate_pairs)
                metrics = compute_metrics(y_true, y_pred)
                row = {
                    "ngram_n": ngram_n,
                    "num_perm": num_perm,
                    "threshold": threshold,
                    "n_docs": int(len(tuning_pair_docs)),
                    "n_pairs": int(len(tuning_pairs)),
                    "n_candidate_pairs": int(len(candidate_pairs)),
                    "insert_sec": insert_sec,
                    "query_sec": query_sec,
                    **metrics
                }
                pair_tuning_rows.append(row)
                print(
                    f"[PAIR] ng={ngram_n} | perm={num_perm} | thr={threshold:.2f} "
                    f"-> P={row['precision']:.4f} R={row['recall']:.4f} F1={row['f1']:.4f}"
                )

    pair_tuning_df = pd.DataFrame(pair_tuning_rows).sort_values(
        ["f1", "precision", "recall"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    print("\n=== Pair tuning top 10 ===")
    display(pair_tuning_df.head(10))

    best_pair_cfg = pair_tuning_df.iloc[0].to_dict()

    pair_test_doc_ids = test_pair_docs["doc_id"].astype(str).tolist()
    pair_test_sig_cache = SignatureCache(
        doc_df=test_pair_docs,
        doc_ids=pair_test_doc_ids,
        ngram_n=int(best_pair_cfg["ngram_n"]),
        num_perm=int(best_pair_cfg["num_perm"]),
        cache_name="test_pair",
    )
    pair_test_sig_cache.load_or_build()

    pair_test_candidate_pairs, insert_sec, query_sec = build_lsh_and_candidate_pairs_from_sigcache(
        sig_cache=pair_test_sig_cache,
        doc_ids=pair_test_doc_ids,
        threshold=float(best_pair_cfg["threshold"]),
    )

    pair_y_true_test, pair_y_pred_test = predict_pairs_from_candidates(test_pairs, pair_test_candidate_pairs)
    pair_test_result = {
        "ngram_n": int(best_pair_cfg["ngram_n"]),
        "num_perm": int(best_pair_cfg["num_perm"]),
        "threshold": float(best_pair_cfg["threshold"]),
        "n_docs": int(len(test_pair_docs)),
        "n_pairs": int(len(test_pairs)),
        "n_candidate_pairs": int(len(pair_test_candidate_pairs)),
        "insert_sec": insert_sec,
        "query_sec": query_sec,
        **compute_metrics(pair_y_true_test, pair_y_pred_test),
    }

    print("\n=== Pair test result ===")
    for k, v in pair_test_result.items():
        print(f"{k}: {v}")

    print("\n=== Classification report (pair test) ===")
    print(classification_report(pair_y_true_test, pair_y_pred_test, target_names=["Negative", "Positive"], zero_division=0))

    pair_family_report = analyze_pair_family_better(test_pairs, pair_y_pred_test)
    positive_pair_type_report = analyze_positive_pair_types(test_pairs, pair_y_pred_test)

    print("\n=== Pair family report ===")
    display(pair_family_report)

    print("\n=== Positive pair type report ===")
    display(positive_pair_type_report)


In [17]:
out_root = Path(PAIR_OUT_ROOT)
out_root.mkdir(parents=True, exist_ok=True)

tuning_docs.to_parquet(out_root / "tuning_docs_optimized.parquet", index=False)
test_docs.to_parquet(out_root / "test_docs_optimized.parquet", index=False)

stream_tuning_df.to_csv(out_root / "stream_tuning_results.csv", index=False)
stream_test_df.to_csv(out_root / "stream_test_results.csv", index=False)

if pair_tuning_df is not None:
    pair_tuning_df.to_csv(out_root / "pair_tuning_results.csv", index=False)
if pair_family_report is not None:
    pair_family_report.to_csv(out_root / "pair_family_report.csv", index=False)
if positive_pair_type_report is not None:
    positive_pair_type_report.to_csv(out_root / "positive_pair_type_report.csv", index=False)

with open(out_root / "best_configs.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "run_profile": RUN_PROFILE,
            "use_gpu_minhash": USE_GPU_MINHASH,
            "cupy_available": CUPY_AVAILABLE,
            "num_parser_variants": NUM_PARSER_VARIANTS,
            "num_trunc_variants": NUM_TRUNC_VARIANTS,
            "prevalence_grid": PREVALENCE_GRID,
            "best_stream_cfg": best_stream_cfg,
            "stream_test_mean_metrics": stream_test_df[["precision", "recall", "f1"]].mean().to_dict(),
            "pair_test_result": pair_test_result,
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved artifacts to:", out_root)


Saved artifacts to: /kaggle/working/mmds_near_dup_optimized
